[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/python-ai-business-data-science/blob/main/fast_track/07_visualization_and_stats.ipynb)

# 📓 Notebook 7 (fast track) — Visualization & Statistics

> **Module:** Data Science Libraries · **Estimated time:** 70–90 min · **Difficulty:** Beginner → Intermediate

Two skills that turn a DataFrame into a *decision*: drawing the chart that makes the pattern obvious, and running the statistic that tells you whether the pattern is real. This notebook keeps the essentials of both — **matplotlib** for charts and the **statistics** toolkit (distributions, confidence intervals, hypothesis tests) — in one lean pass. It builds directly on pandas (Notebook 6).


> 🏎️ **You're on the fast track.** This is a trimmed, **combined** notebook that condenses two canonical chapters into one: [`02_data_science/09_matplotlib_basics.ipynb`](../02_data_science/09_matplotlib_basics.ipynb) (Matplotlib) and [`02_data_science/10_statistics_basics.ipynb`](../02_data_science/10_statistics_basics.ipynb) (Statistics). From each, the Stretch exercises (A–D) and the 🎁 Bonus mini-project have been removed to keep the path lean — the core teaching, every Practice exercise, and Stretch C/D are kept. Open the canonical versions once you want the deeper material.

---


# 📊 Part 1 — Visualization with Matplotlib  *(canonical NB 9)*


## 1. Setup — the canonical imports

Just three lines, and they are the same in every notebook you'll ever see.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A consistent, professional look. Run once at the top of any notebook.
plt.rcParams.update({
    "figure.figsize"   : (8, 5),
    "figure.dpi"       : 100,
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.size"        : 11,
})

print(f"matplotlib version: {plt.matplotlib.__version__}")


> 💡 **rcParams.** Setting these at the top of a notebook gives every figure a consistent look. Tweaking the default style is the single biggest *return-per-line* investment you can make in your plotting code.

## 2. The Figure / Axes mental model

Matplotlib has two layers you need to keep straight:

```
┌────────────── Figure (the whole window) ──────────────┐
│                                                        │
│   ┌────── Axes (one plot inside) ──────┐               │
│   │                                    │               │
│   │   data lives here                  │               │
│   │                                    │               │
│   └────────────────────────────────────┘               │
│                                                        │
└────────────────────────────────────────────────────────┘
```

A **Figure** is the canvas; an **Axes** is one plot inside it. You can have many Axes in one Figure. We always use the object-oriented form `fig, ax = plt.subplots(...)` — it scales smoothly from "one plot" to "twelve-panel dashboard".

## 3. Your first line plot — automation rate over a year

We'll use the same support-operations theme as NB7 throughout. First, a single line plot of monthly automation rate.

In [ ]:
months = np.arange(1, 13)
auto_rate = np.array([0.55, 0.58, 0.60, 0.63, 0.65, 0.68, 0.70, 0.72, 0.74, 0.76, 0.78, 0.81])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(months, auto_rate, color="#4C72B0", linewidth=2, marker="o")
ax.set_title("Monthly automation rate — Chat channel")
ax.set_xlabel("Month")
ax.set_ylabel("Automation rate")
ax.set_xticks(months)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


The seven lines above are essentially every plot you'll ever make:

1. `fig, ax = plt.subplots(...)` — create a Figure and one Axes.
2. `ax.plot(...)` — draw the data.
3. `ax.set_title / set_xlabel / set_ylabel` — label it.
4. `ax.set_xticks / set_ylim / ...` — adjust axes when needed.
5. `plt.tight_layout()` — fix overlapping labels.
6. `plt.show()` — render.

Everything else is variations on this recipe.

## 4. Multiple lines, legends, line styles

Same x-axis, multiple lines — perfect for "the same metric across categories".

In [ ]:
months = np.arange(1, 13)
chat     = np.array([0.55, 0.58, 0.60, 0.63, 0.65, 0.68, 0.70, 0.72, 0.74, 0.76, 0.78, 0.81])
email    = np.array([0.42, 0.45, 0.47, 0.50, 0.52, 0.55, 0.57, 0.60, 0.62, 0.65, 0.67, 0.70])
phone    = np.array([0.15, 0.16, 0.17, 0.18, 0.20, 0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 0.28])
web_form = np.array([0.62, 0.65, 0.68, 0.70, 0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.83, 0.85])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(months, chat,     label="Chat",     color="#55A467", linewidth=2, marker="o")
ax.plot(months, email,    label="Email",    color="#4C72B0", linewidth=2, marker="s")
ax.plot(months, phone,    label="Phone",    color="#C44E52", linewidth=2, marker="^", linestyle="--")
ax.plot(months, web_form, label="Web Form", color="#DD8452", linewidth=2, marker="d")

ax.set_title("Automation rate by channel (one year)")
ax.set_xlabel("Month")
ax.set_ylabel("Automation rate")
ax.set_xticks(months)
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


> 💡 **Common line styles:** `"-"` solid, `"--"` dashed, `":"` dotted, `"-."` dash-dot. **Markers:** `"o"` circle, `"s"` square, `"^"` triangle, `"d"` diamond, `"x"` cross, `"."` point.

For colour, use hex codes (`"#4C72B0"`), CSS names (`"crimson"`), or any standard matplotlib colour. **Stick to a small, consistent palette** — your audience will read the chart faster.

## 5. Scatter plots — relationships between two variables

For *"is X related to Y?"* use a scatter. We'll plot cost vs satisfaction for a batch of calls, coloured by the customer segment.

In [ ]:
rng = np.random.default_rng(42)
n = 80

# Mock: cost driven by tokens, satisfaction mildly inversely related to cost
cost          = rng.uniform(0.001, 0.012, size=n)
satisfaction  = 4.5 - 80 * cost + rng.normal(0, 0.25, size=n)
segment_idx   = rng.integers(0, 3, size=n)
segment_names = np.array(["SMB", "Mid-market", "Enterprise"])
colours       = np.array(["#4C72B0", "#DD8452", "#55A467"])

fig, ax = plt.subplots(figsize=(8, 5))

# One scatter per segment — keeps the legend tidy
for i, name in enumerate(segment_names):
    sel = segment_idx == i
    ax.scatter(cost[sel], satisfaction[sel],
               c=colours[i], s=55, edgecolor="black", alpha=0.8, label=name)

# A simple linear fit on top — communicates the trend
m, b = np.polyfit(cost, satisfaction, 1)
xs = np.linspace(cost.min(), cost.max(), 50)
ax.plot(xs, m*xs + b, "k--", linewidth=1.4, label=f"fit: slope={m:+.1f}")

ax.set_title("Cost per call vs customer satisfaction")
ax.set_xlabel("Cost per call (USD)")
ax.set_ylabel("Satisfaction (1–5)")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the chart.** The downward slope of the trend line says higher-cost calls tend to come with slightly lower satisfaction — possibly because expensive calls are the hard ones the bot couldn't fully handle. The colour encoding lets you check whether *segment* explains anything beyond cost (it mostly doesn't here — the segments overlap heavily).

## 6. Bar charts — comparing categories

In [ ]:
channels = ["Email", "Chat", "Phone", "Web Form", "Social"]
annual_cost_usd = [9_400, 14_200, 11_800, 6_700, 3_900]
palette = ["#4C72B0", "#55A467", "#C44E52", "#DD8452", "#8172B2"]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(channels, annual_cost_usd, color=palette, edgecolor="black")

# Annotate each bar with its value — never make people read off the y-axis
for bar, val in zip(bars, annual_cost_usd):
    ax.text(bar.get_x() + bar.get_width()/2, val, f"${val:,}",
            ha="center", va="bottom", fontsize=10)

ax.set_title("Annual support-channel spend")
ax.set_ylabel("Spend (USD)")
ax.set_ylim(0, max(annual_cost_usd) * 1.15)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Horizontal bar charts

Use `barh` whenever the category labels are long — they read more comfortably horizontally.

In [ ]:
skills = ["Python", "Pandas", "SQL", "Statistics", "Machine Learning",
          "Prompt engineering", "Cloud ops", "Communication"]
importance = [95, 90, 88, 92, 85, 87, 78, 90]

order = np.argsort(importance)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(np.array(skills)[order], np.array(importance)[order],
        color="#4C72B0", edgecolor="black")
ax.set_title("Most useful skills for a modern AI-data role (survey)")
ax.set_xlabel("Self-reported importance (0–100)")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()


## 7. Histograms — distributions

A histogram tells you the *shape* of one variable: is it bell-shaped, skewed, bimodal? Crucial before you start modelling.

In [ ]:
rng = np.random.default_rng(0)

# Two simulated latency distributions
fast_model = rng.normal(loc=1800, scale=400,  size=400)
slow_model = rng.normal(loc=2700, scale=700,  size=400)
fast_model = np.clip(fast_model, 200, None)
slow_model = np.clip(slow_model, 200, None)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(fast_model, bins=25, alpha=0.65, label="Model A (fast)",
        color="#4C72B0", edgecolor="black")
ax.hist(slow_model, bins=25, alpha=0.65, label="Model B (slow)",
        color="#DD8452", edgecolor="black")

ax.axvline(fast_model.mean(), color="#4C72B0", linestyle="--", linewidth=1.5)
ax.axvline(slow_model.mean(), color="#DD8452", linestyle="--", linewidth=1.5)

ax.set_title("Latency distribution — two model candidates")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Number of calls")
ax.legend()
plt.tight_layout()
plt.show()


**Reading a histogram.** The x-axis is the value, the y-axis the count of observations in each bin. Two overlapping histograms with `alpha ≈ 0.65` is a clean way to compare two distributions. Vertical dashed lines highlight the means — much more informative than a single number.

## 8. Box plots — distribution at a glance

When you have *several* distributions to compare side by side, a box plot is denser than a row of histograms.

In [ ]:
rng = np.random.default_rng(1)
models = ["gpt-4o-mini", "claude-haiku", "internal-llm", "open-source-q4"]
samples = [
    rng.normal(1800, 400, 60),
    rng.normal(2400, 600, 60),
    rng.normal(1500, 350, 60),
    rng.normal(2100, 800, 60),
]
samples = [np.clip(s, 200, None) for s in samples]

fig, ax = plt.subplots(figsize=(8, 5))
box = ax.boxplot(samples, tick_labels=models, patch_artist=True,
                 medianprops=dict(color="black", linewidth=2))

palette = ["#4C72B0", "#DD8452", "#55A467", "#8172B2"]
for patch, c in zip(box["boxes"], palette):
    patch.set_facecolor(c); patch.set_alpha(0.75)

ax.set_title("Latency distribution per candidate model")
ax.set_ylabel("Latency (ms)")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


**Reading a box plot.** The box covers the interquartile range (Q1 to Q3). The line inside is the **median**. Whiskers extend to data within 1.5×IQR; points beyond are outliers (drawn as dots). A wide box = high variance, a tight box = consistent performance.

## 9. Multi-panel layouts — `subplots(rows, cols)`

This is the building block of every dashboard you'll ever make.

In [ ]:
rng = np.random.default_rng(7)

# Pretend we have 300 calls and three "channels" of information about them
n = 300
cost       = rng.gamma(2.0, 0.001, size=n)         # right-skewed
satisfaction = rng.normal(4.0, 0.5, size=n).clip(1, 5)
day        = np.arange(n)
volume     = np.cumsum(rng.normal(0, 1, n)) + 50    # cumulative trend

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle("Mini AI-ops dashboard", fontsize=15, fontweight="bold")

# (0, 0) Histogram of cost per call
axes[0, 0].hist(cost * 100, bins=25, color="#4C72B0", edgecolor="black")
axes[0, 0].set_title("Cost per call (¢)")
axes[0, 0].set_xlabel("Cost (cents)")
axes[0, 0].set_ylabel("Number of calls")

# (0, 1) Scatter of cost vs satisfaction
axes[0, 1].scatter(cost * 100, satisfaction, alpha=0.4, color="#DD8452",
                   edgecolor="black", s=20)
axes[0, 1].set_title("Cost vs satisfaction")
axes[0, 1].set_xlabel("Cost (cents)")
axes[0, 1].set_ylabel("Satisfaction (1–5)")

# (1, 0) Line trend
axes[1, 0].plot(day, volume, color="#55A467", linewidth=1.6)
axes[1, 0].set_title("Daily volume index (cumulative)")
axes[1, 0].set_xlabel("Day of year")
axes[1, 0].set_ylabel("Volume index")

# (1, 1) Box plot of satisfaction by quarter
quarters = ["Q1", "Q2", "Q3", "Q4"]
groups = [satisfaction[(day >= i*75) & (day < (i+1)*75)] for i in range(4)]
axes[1, 1].boxplot(groups, tick_labels=quarters, patch_artist=True)
axes[1, 1].set_title("Satisfaction by quarter")
axes[1, 1].set_ylabel("Satisfaction (1–5)")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


> 💡 `axes` is a NumPy array of Axes objects — `axes[0, 0]` is top-left, `axes[1, 1]` is bottom-right. For a single row or column you get a 1-D array (`axes[0]`, `axes[1]`).

This 2×2 layout is the **canonical "executive dashboard"** shape — and it's exactly the layout you'll use in the NB 24 capstone. Get comfortable with it.

## 10. Annotations — making the punchline visible

A *good* chart leaves no ambiguity about what the reader should look at.

In [ ]:
months = np.arange(1, 25)
auto_rate = np.array([
    0.40, 0.42, 0.45, 0.46, 0.48, 0.50, 0.51, 0.52, 0.53, 0.55, 0.56, 0.58,
    0.60, 0.62, 0.65, 0.55,    # bug regression in month 16
    0.66, 0.69, 0.71, 0.73, 0.74, 0.76, 0.77, 0.79,
])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(months, auto_rate, marker="o", color="#4C72B0", linewidth=2)

ax.annotate("Regression\n(post-deploy bug)",
            xy=(16, 0.55), xytext=(13, 0.42),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=10)

ax.annotate("Quarterly review",
            xy=(24, 0.79), xytext=(20, 0.66),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=10)

ax.set_title("Automation rate over two years")
ax.set_xlabel("Month")
ax.set_ylabel("Automation rate")
ax.set_ylim(0.3, 0.9)
plt.tight_layout()
plt.show()


## 11. Heatmaps — visualising matrices

Heatmaps are perfect for correlation matrices, confusion matrices, or any "every-row-by-every-column" comparison.

In [ ]:
# A small confusion-matrix-ish view: predicted vs actual sentiment
labels = ["positive", "neutral", "negative"]
cm = np.array([
    [42, 3,  1],     # true positive
    [ 4, 38, 5],     # true neutral
    [ 1, 6, 35],     # true negative
])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(3), labels)
ax.set_yticks(range(3), labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Sentiment-classifier confusion matrix")

# Annotate every cell
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j],
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
                fontsize=12)

fig.colorbar(im, ax=ax, label="Number of examples")
plt.tight_layout()
plt.show()


**This exact plot is what you'll use in NB 14** to evaluate a real classifier. The diagonal shows correct predictions; off-diagonal cells show where the model is confused.

## 12. Saving figures

Drop `plt.savefig("name.png", dpi=200, bbox_inches="tight")` *before* `plt.show()` to save your figure to disk. Common formats:

| Extension | Use for                                              |
|-----------|------------------------------------------------------|
| `.png`    | reports, web, default raster                          |
| `.pdf`    | papers, slides, infinitely scalable vector            |
| `.svg`    | web, when you may edit later in Illustrator/Inkscape  |

```python
fig, ax = plt.subplots()
ax.plot(x, y)
plt.savefig("automation_rate.png", dpi=200, bbox_inches="tight")
plt.show()
```

## 13. Common pitfalls

| Pitfall                                       | Symptom                                | Fix |
|-----------------------------------------------|----------------------------------------|-----|
| Squished figure / overlapping labels          | crowded titles                         | call `plt.tight_layout()` before `plt.show()` |
| Multiple plots in one cell mixed by accident  | extra empty axes                       | always open a new figure with `plt.subplots(...)` |
| No axis labels / units                        | viewer can't tell what is plotted      | always set title + xlabel + ylabel |
| 3-D pie charts and other chartjunk            | takes longer to read                   | almost never use them; bar > pie > 3-D pie |
| Hard-to-distinguish colours                   | viewers can't tell categories apart    | use a small consistent palette; consider colourblind-safe colormaps |

## 🧪 Practice exercises

### Exercise 1 — ⭐ Three lines, one plot

On the interval `x ∈ [0, 5]`, plot `y₁ = x`, `y₂ = x²`, `y₃ = 2ˣ` on the same axes. Add labels, a legend, a title, and a grid.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
x = np.linspace(0, 5, 200)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, x,     label="y = x",   linewidth=2)
ax.plot(x, x**2,  label="y = x²",  linewidth=2)
ax.plot(x, 2**x,  label="y = 2ˣ",  linewidth=2)

ax.set_title("Linear vs quadratic vs exponential growth")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()
```

A useful intuition for AI cost models: exponential growth (e.g., context length × cost) overtakes anything else *eventually* — visible immediately from a chart like this.
</details>

### Exercise 2 — ⭐⭐ Histogram with mean line

Generate 1,000 simulated latencies from a normal distribution with mean 2000 ms and std 400 ms. Plot a histogram, overlay vertical dashed lines for the mean and (mean ± std).

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
rng = np.random.default_rng(0)
data = rng.normal(2000, 400, 1000)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(data, bins=30, color="#4C72B0", edgecolor="black", alpha=0.8)

m, s = data.mean(), data.std()
ax.axvline(m,     color="red",    linestyle="--", label=f"mean = {m:.0f}")
ax.axvline(m - s, color="orange", linestyle=":")
ax.axvline(m + s, color="orange", linestyle=":", label=f"±1 std = {s:.0f}")

ax.set_title("Simulated latency distribution")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Number of calls")
ax.legend()
plt.tight_layout()
plt.show()
```
</details>

### Exercise 3 — ⭐⭐ Subplots side-by-side

Create a figure with **two side-by-side subplots**:

1. Left: a bar chart of monthly spend by channel (sample data below).
2. Right: a pie chart of the same data.

Use the same colour palette for both, and add `fig.suptitle(...)`.

In [ ]:
# Your code here  👇
channels = ["Email", "Chat", "Phone", "Web Form"]
spend    = [12_200, 9_500, 14_500, 8_800]


<details>
<summary>💡 <b>Solution</b></summary>

```python
palette = ["#4C72B0", "#DD8452", "#55A467", "#C44E52"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle("Monthly support-channel spend", fontsize=14, fontweight="bold")

# Bar
axes[0].bar(channels, spend, color=palette, edgecolor="black")
axes[0].set_title("Bar")
axes[0].set_ylabel("Spend (USD)")
axes[0].grid(axis="y", alpha=0.3)

# Pie
axes[1].pie(spend, labels=channels, colors=palette,
            autopct="%1.0f%%", startangle=90, wedgeprops=dict(edgecolor="white"))
axes[1].set_title("Pie")

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()
```

A pie chart is *only* a good choice when you have ≤ 5 categories and absolute proportions are the point. For ranking categories, a bar chart wins every time.
</details>

### Exercise 4 — ⭐⭐ Annotated scatter

Generate 100 random (cost, satisfaction) points where `satisfaction = 4.5 − 100·cost + noise`. Plot them, add the regression line, and annotate the **largest residual** (the point furthest from the line).

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
rng = np.random.default_rng(3)
cost = rng.uniform(0.001, 0.020, 100)
sat  = 4.5 - 100 * cost + rng.normal(0, 0.3, 100)

m, b = np.polyfit(cost, sat, 1)
xs   = np.linspace(cost.min(), cost.max(), 50)
pred = m * cost + b
res  = sat - pred
i    = np.argmax(np.abs(res))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(cost, sat, alpha=0.6, color="#4C72B0", s=30, edgecolor="black")
ax.plot(xs, m*xs + b, "r--", linewidth=1.5, label=f"fit: slope={m:.0f}")

ax.scatter(cost[i], sat[i], color="red", s=120, edgecolor="black", zorder=3)
ax.annotate(f"largest residual\n(res = {res[i]:+.2f})",
            xy=(cost[i], sat[i]),
            xytext=(cost[i] + 0.002, sat[i] - 0.5),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=10)

ax.set_title("Cost vs satisfaction with largest outlier marked")
ax.set_xlabel("Cost per call (USD)")
ax.set_ylabel("Satisfaction (1–5)")
ax.legend()
plt.tight_layout()
plt.show()
```

Highlighting an outlier directly on the chart is one of the highest-leverage things you can do with a plot — it makes the next question (*"why?"*) jump out.
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞

The plot below should show *two* sine curves with different phases. It currently only shows one and has no axis labels. Fix it.

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
x = np.linspace(0, 2*np.pi, 200)

plt.figure()
plt.plot(x, np.sin(x))
plt.plot(x, np.sin(x))       # bug: same curve again
plt.title("Two sine curves")
plt.show()


<details>
<summary>💡 <b>Solution</b></summary>

Two issues: both lines plot the same `sin(x)`, and the axes are unlabelled.

```python
x = np.linspace(0, 2*np.pi, 200)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, np.sin(x),         label="sin(x)")
ax.plot(x, np.sin(x + np.pi/4), label="sin(x + π/4)")
ax.set_title("Two sine curves with a phase shift")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()
```

A useful habit: never trust a chart you just drew. Look at it, ask "would a stranger understand this in 5 seconds?", and fix anything that fails the test.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise C — ⭐⭐⭐ Two panels, shared x-axis

Plot two related signals on top of each other in a single figure with two stacked subplots that share the x-axis. Upper panel: a sine wave. Lower panel: its derivative (a cosine). Label each axis and give the figure a single title.

Hint: `plt.subplots(2, 1, sharex=True)`.

In [ ]:
# Your code here  👇
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 4*np.pi, 200)

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 4*np.pi, 200)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9, 4))
ax1.plot(x, np.sin(x), label="sin(x)")
ax1.set_ylabel("signal")
ax1.legend(loc="upper right")

ax2.plot(x, np.cos(x), color="C1", label="d/dx sin(x) = cos(x)")
ax2.set_xlabel("x")
ax2.set_ylabel("derivative")
ax2.legend(loc="upper right")

fig.suptitle("A signal and its derivative")
fig.tight_layout()
plt.show()
```

**Reasoning.** Three matplotlib habits this exercise builds. (1) Always use the **object-oriented API** (`fig, (ax1, ax2) = plt.subplots(...)`) rather than the pyplot state machine — when you have multiple axes, `plt.plot()` becomes ambiguous about which one you're drawing on. (2) `sharex=True` keeps the x-axes aligned, which is exactly what you want when the bottom panel is a derived view of the top. (3) `fig.tight_layout()` saves you from manually fiddling with subplot spacing; together with `fig.suptitle(...)` it's the right way to give the whole figure a single title.
</details>

### Stretch exercise D — ⭐⭐⭐ Annotate the peak of a curve

Plot a noisy curve (`y = sin(x) + small noise`) and add an arrow annotation pointing at its global maximum, labelled with the (x, y) coordinates of the peak. Use `ax.annotate` with `arrowprops`.

Hint: `np.argmax` gives you the index of the maximum.

In [ ]:
# Your code here  👇
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = np.linspace(0, 4*np.pi, 200)
y = np.sin(x) + 0.1 * rng.standard_normal(x.shape)

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = np.linspace(0, 4*np.pi, 200)
y = np.sin(x) + 0.1 * rng.standard_normal(x.shape)

i = int(np.argmax(y))
peak_x, peak_y = float(x[i]), float(y[i])

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(x, y, color="C0")
ax.scatter([peak_x], [peak_y], color="red", zorder=5)
ax.annotate(
    f"peak\n(x={peak_x:.2f}, y={peak_y:.2f})",
    xy=(peak_x, peak_y),
    xytext=(peak_x + 1.5, peak_y - 0.4),
    arrowprops=dict(arrowstyle="->", color="red"),
    fontsize=10, color="red",
)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_title("Noisy sine — annotated peak")
plt.tight_layout()
plt.show()
```

**Reasoning.** Annotations are how plots tell a story without forcing the reader to squint. Three details worth absorbing. (1) `xy=...` is the point you're pointing at; `xytext=...` is where the label sits — separate them by some offset so the arrow is visible. (2) `zorder=5` pushes the red dot above the line so it isn't hidden. (3) Always compute the peak from the data (`np.argmax`) rather than hard-coding coordinates — the figure stays correct if you regenerate the data with a different random seed.
</details>

## 🧠 Key takeaways

1. Use the **Figure / Axes** (`fig, ax = plt.subplots()`) style for everything beyond a one-off.
2. Always label your axes and give the plot a title — a chart without context is a riddle.
3. Match the **chart type** to the question:
   - **line** for *trend over time*,
   - **bar** for *comparing categories*,
   - **histogram** / **box** for *distributions*,
   - **scatter** for *relationships*,
   - **heatmap** for *matrices*.
4. A 2×2 **dashboard** (`subplots(2, 2)`) is enough for most executive summaries.
5. Annotate the punchline directly on the chart — don't make the reader hunt for it.
6. Use a small, consistent **palette**.
7. Save with `plt.savefig("name.png", dpi=200, bbox_inches="tight")` for reports.

## ✅ Self-assessment

- [ ] Build a single-panel plot with title, axis labels, and a legend
- [ ] Compare multiple categories with bar / horizontal-bar charts
- [ ] Show a distribution with a histogram and a box plot
- [ ] Show a relationship with a scatter and an overlaid regression line
- [ ] Build a 2×2 subplots layout with a shared `suptitle`
- [ ] Add an annotation with an arrow to the right place on the chart
- [ ] Save a chart to PNG at publication DPI

# 📈 Part 2 — Statistics Basics  *(canonical NB 10)*


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

RNG = np.random.default_rng(seed=42)


## 2. Two latency datasets — see them before you summarise them

We will use two simulated batches of LLM call latencies (in ms) throughout the notebook:

- **Model A** — a *fast* model. Mean ~1800 ms.
- **Model B** — a *slow* model with occasional spikes. Mean ~2400 ms.

In [ ]:
# Two latency distributions (simulated). Model B has occasional spikes.
n = 200
latency_a = RNG.normal(loc=1800, scale=300, size=n).clip(min=200)

# Mixture: 90% normal, 10% slow outliers
mask = RNG.random(n) < 0.9
latency_b = np.where(mask,
                     RNG.normal(2300, 350, n),
                     RNG.normal(4500, 600, n)).clip(min=200)

print(f"Model A:  n={len(latency_a)}, mean={latency_a.mean():.0f}, std={latency_a.std(ddof=1):.0f}")
print(f"Model B:  n={len(latency_b)}, mean={latency_b.mean():.0f}, std={latency_b.std(ddof=1):.0f}")


### Visualise — *always* before you summarise

A single chart often tells you more than a table of summary statistics. Histograms show the shape; box plots show the centre and tails.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (1) Histograms
ax = axes[0]
ax.hist(latency_a, bins=30, alpha=0.6, label="Model A", color="#4C72B0", edgecolor="black")
ax.hist(latency_b, bins=30, alpha=0.6, label="Model B", color="#DD8452", edgecolor="black")
ax.axvline(latency_a.mean(), color="#4C72B0", lw=2, ls="--")
ax.axvline(latency_b.mean(), color="#DD8452", lw=2, ls="--")
ax.set_title("Histograms with means")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Count")
ax.legend()

# (2) Box plots
ax = axes[1]
bp = ax.boxplot([latency_a, latency_b], tick_labels=["Model A", "Model B"],
                patch_artist=True)
for patch, c in zip(bp["boxes"], ["#4C72B0", "#DD8452"]):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_title("Box plots — centre & tails at a glance")
ax.set_ylabel("Latency (ms)")
plt.tight_layout(); plt.show()


**Reading what you see:**

- Model A is a clean bell-curve. Mean ≈ median, light tails, no outliers.
- Model B has a **right-skewed** distribution. The mean is dragged up by the spikes. The box plot shows those spikes as outlier dots above the upper whisker.

> 🎯 **The first statistical decision of any analysis** is whether the mean is a fair summary. When the distribution is skewed or has fat tails, the median is often more honest.

## 3. Mean vs median — when each one lies

In [ ]:
for name, data in [("Model A", latency_a), ("Model B", latency_b)]:
    print(f"{name}:")
    print(f"  mean   = {np.mean(data):>7.1f} ms")
    print(f"  median = {np.median(data):>7.1f} ms")
    print(f"  std    = {np.std(data, ddof=1):>7.1f} ms")
    p99 = np.percentile(data, 99)
    print(f"  p99    = {p99:>7.1f} ms     (top 1% of users wait this long or longer)")
    print()


**Why this matters operationally.** If you tell your team "Model B's average latency is 2.5 seconds" but its **p99 latency is 5 seconds**, you're under-reporting the real user experience for 1% of your traffic.

> 💡 **Always report p50, p95, p99 alongside the mean** for any latency / cost / waiting-time metric. The "tail" is where customers leave.

## 4. Confidence intervals — how sure are you of that mean?

The mean of a sample is a *point estimate*. A different sample would have given a slightly different mean. A **confidence interval** quantifies that wiggle room.

```
   95% CI for the mean of Model A:  [1761 ms, 1845 ms]
                                     │              │
                                     │              └─ upper bound
                                     └────────────── lower bound
```

The 95% CI says: *"if I repeated this experiment many times, 95% of the constructed intervals would contain the true mean."*

(That's the technically correct interpretation. Casually, "we're 95% sure the true mean is in here" is close enough for most business conversations.)

In [ ]:
def ci_of_mean(data, confidence=0.95):
    """Return (lower, upper) of the confidence interval for the mean."""
    n = len(data)
    mean = np.mean(data)
    sem  = np.std(data, ddof=1) / np.sqrt(n)        # standard error of the mean
    # stats.t.ppf(p, df) is the inverse CDF: the t-value with fraction p of the
    # distribution below it. Here 0.975 gives the ± critical multiplier for 95%.
    crit = stats.t.ppf(0.5 + confidence/2, df=n-1)
    return mean - crit*sem, mean + crit*sem


for name, data in [("Model A", latency_a), ("Model B", latency_b)]:
    lo, hi = ci_of_mean(data, confidence=0.95)
    print(f"{name}:  mean = {data.mean():.1f} ms,  95% CI = [{lo:.1f}, {hi:.1f}]  "
          f"(±{(hi-lo)/2:.1f})")


**Three things to notice:**

1. Both CIs have a "half-width" of around 40–70 ms — the *uncertainty* on a 200-point mean.
2. The two CIs don't overlap → very strong evidence the means are *actually* different.
3. **Larger samples → narrower CIs.** Halving the half-width takes 4× the data.

> 🎯 **Rule of thumb for fast back-of-envelope CIs:** for n=200 normal-ish data, the half-width is roughly the standard deviation ÷ 7. (Comes from `1.96/sqrt(200) ≈ 0.14`.)

## 5. The t-test — is the difference *real*?

When you measure two things (Model A, Model B) and see different means, you need to know whether the difference is **bigger than the noise**. The classic tool is **Welch's t-test** (the safer of the two t-test flavours — doesn't assume equal variances).

In [ ]:
# Two-sample, two-sided Welch's t-test
result = stats.ttest_ind(latency_a, latency_b, equal_var=False)
print(f"t-statistic = {result.statistic:.2f}")
print(f"p-value     = {result.pvalue:.3e}")

# Mean difference and its 95% CI
diff = latency_b.mean() - latency_a.mean()
se = np.sqrt(np.var(latency_a, ddof=1) / len(latency_a) +
             np.var(latency_b, ddof=1) / len(latency_b))
ci_lo, ci_hi = diff - 1.96 * se, diff + 1.96 * se
print(f"\nMean difference (B − A) = {diff:.1f} ms")
print(f"95% CI for the difference: [{ci_lo:.1f}, {ci_hi:.1f}]")


### How to read the output

- **t-statistic ≈ 24** means the difference of means is 24 standard errors away from zero — *enormous*.
- **p-value ≈ 10⁻⁶⁰** means: *"if the two models actually had the same mean latency, we'd see this big a difference by chance in fewer than 1 in 10⁶⁰ experiments."*
- **The CI for the difference excludes 0** → the difference is real.

> ⚠️ **Common p-value misunderstandings:**
>
> - p < 0.05 does **not** mean "95% chance the result is real" (a Bayesian probability).
> - p < 0.05 does **not** mean "the difference is important" — it might be 2 ms, statistically significant, and operationally meaningless.
> - p < 0.05 is **not** a free pass: with enough data, *every* tiny difference becomes "significant".
>
> The cure for all three: **always report effect size alongside p-value**.

## 6. Effect size — the number managers care about

The p-value tells you whether a difference is real. **Cohen's *d*** tells you how *big* the difference is, in units of the natural spread of the data.

$$d = \frac{\bar{x}_B - \bar{x}_A}{s_\text{pooled}}$$

| Cohen's *d* | Rough label |
|---|---|
| 0.2 | small |
| 0.5 | medium |
| 0.8 | large |
| > 1.0 | very large |

In [ ]:
def cohens_d(x, y):
    """Cohen's d for two independent samples."""
    nx, ny = len(x), len(y)
    pooled = np.sqrt(((nx-1)*np.var(x, ddof=1) + (ny-1)*np.var(y, ddof=1)) / (nx + ny - 2))
    return (np.mean(y) - np.mean(x)) / pooled


d = cohens_d(latency_a, latency_b)
print(f"Cohen's d = {d:+.2f}")
print(f"Mean diff = {latency_b.mean() - latency_a.mean():.0f} ms")


**How to report this to a manager:**

> *"Model B is roughly 500 ms slower than Model A on average (Cohen's d ≈ 1.4, very large effect). 95% CI for the difference: [460, 560] ms. p < 10⁻⁵⁹."*

The numbers in **bold below** are the ones the manager will remember:

- **how big** is the difference (Cohen's d, mean diff)
- **how sure** are we (CI, p-value)
- **what does it mean in their units** (500 ms — would users notice?)

## 7. Sample size — how much data do I need?

Before you run a 10,000-user A/B test, ask: *"what's the smallest sample that would let me detect the effect I care about, with 80% power, at significance 0.05?"*

The formula (for two equal groups, two-sided t-test, normal data) is roughly:

$$n \approx \frac{16}{d^2}$$

per group, where `d` is the *minimum* effect size you'd care about.

In [ ]:
def sample_size_for_d(d, alpha=0.05, power=0.80):
    """Approximate per-group sample size to detect Cohen's d with given power."""
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta  = stats.norm.ppf(power)
    return int(np.ceil(2 * ((z_alpha + z_beta) / d) ** 2))


print(f"{'effect':>8}  {'per-group n':>12}")
for d in [0.1, 0.2, 0.5, 0.8, 1.0]:
    n = sample_size_for_d(d)
    print(f"  d = {d:.1f}    {n:>10,}")


**The big takeaway.** Tiny effects need *enormous* samples. If you can only run 200 users per arm, you can reliably detect a Cohen's d of about 0.3 — but anything smaller will hide in the noise.

> 💡 **Sample-size planning prevents the silent failure** where you ran a test, the p-value was 0.12, and you concluded "no effect" — when you actually just didn't have enough data.

## 8. A complete A/B-test report

The "good" version of A/B reporting puts all five numbers a manager wants on one screen:

In [ ]:
def ab_report(name_a, x, name_b, y, alpha=0.05):
    """Print a five-number A/B summary that's safe to send to a manager."""
    n_a, n_b   = len(x), len(y)
    mean_a, mean_b = np.mean(x), np.mean(y)
    diff       = mean_b - mean_a
    var_a, var_b = np.var(x, ddof=1)/n_a, np.var(y, ddof=1)/n_b
    se_diff    = np.sqrt(var_a + var_b)
    # Welch–Satterthwaite df, so the CI uses the SAME t-distribution as the
    # Welch p-value below — otherwise the CI and the p-value could disagree.
    df_welch   = (var_a + var_b)**2 / (var_a**2/(n_a-1) + var_b**2/(n_b-1))
    t_crit     = stats.t.ppf(1 - alpha/2, df_welch)
    ci_lo, ci_hi = diff - t_crit*se_diff, diff + t_crit*se_diff
    d          = cohens_d(x, y)
    p_val      = stats.ttest_ind(x, y, equal_var=False).pvalue

    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f" A/B test: {name_a}  vs  {name_b}")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f" Sample sizes      : {n_a} vs {n_b}")
    print(f" Mean              : {mean_a:.1f}  vs  {mean_b:.1f}")
    print(f" Difference (B−A)  : {diff:+.1f}")
    print(f" 95% CI for diff   : [{ci_lo:+.1f}, {ci_hi:+.1f}]")
    print(f" Cohen's d         : {d:+.2f}     (effect size)")
    print(f" p-value           : {p_val:.2e}")
    verdict = "REAL difference" if p_val < alpha and abs(d) > 0.2 \
              else "NOT meaningfully different"
    print(f" Verdict           : {verdict}")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")


ab_report("Model A", latency_a, "Model B", latency_b)


**Why this template works.**

- **Sample sizes** show whether the test was powered enough to trust the verdict.
- **Means + difference + CI** answer "by how much?" — the manager's first question.
- **Cohen's d** answers "is the difference *big* in real-world terms?" — the second question.
- **p-value** answers "is it real?" — the third question.
- The **verdict** combines the two ("real *and* meaningful"), preventing the common mistake of acting on a statistically significant but operationally tiny effect.

## 9. The three classical mistakes (and the fixes)

| Mistake | What happens | Fix |
|---|---|---|
| **Peeking** — running a test, looking at the p-value daily, stopping the first time it dips below 0.05 | False positives explode. With 5 peeks the *actual* false-positive rate is around 20%, not 5%. | Plan the sample size in advance. Stop *only* at that pre-planned moment. |
| **Multiple comparisons** — running 20 t-tests on 20 metrics, declaring the one with p<0.05 as a "win" | At alpha=0.05 you expect 1 false positive per 20 tests by definition. | Use Bonferroni (alpha/k) or false-discovery-rate correction. Or: pre-register your hypothesis. |
| **Reporting only p, not d** | Statistically significant but operationally meaningless effects get shipped. | Always report effect size alongside p-value. |

## 🧪 Practice exercises

### Exercise 1 — ⭐ Robust statistics on a heavy-tailed metric

Compute the **mean**, **median**, **5%-trimmed mean**, and **standard deviation** of `latency_b`. Which one would you report to a manager who asks "what's a typical latency?".

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
from scipy.stats import trim_mean

print(f"mean              = {np.mean(latency_b):.1f}")
print(f"median            = {np.median(latency_b):.1f}")
print(f"5%-trimmed mean   = {trim_mean(latency_b, 0.05):.1f}")
print(f"std               = {np.std(latency_b, ddof=1):.1f}")
```

The **median** is usually the most honest "typical user" number on a right-skewed
metric like latency. The mean and std are dragged up by the slow tail. The
trimmed mean (drop the top and bottom 5% before averaging) is a robust compromise.
</details>

### Exercise 2 — ⭐⭐ Confidence interval, by hand

Without using `stats.t.ppf`, compute an *approximate* 95% CI for the mean of `latency_a` using the rule of thumb `mean ± 1.96 × std/√n`. Compare with the t-distribution-based CI from §4.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
n = len(latency_a)
mean = latency_a.mean()
sem  = latency_a.std(ddof=1) / np.sqrt(n)

print(f"Normal-approx CI :  [{mean - 1.96*sem:.1f},  {mean + 1.96*sem:.1f}]")
print(f"t-distribution CI:  [{ci_of_mean(latency_a)[0]:.1f},  {ci_of_mean(latency_a)[1]:.1f}]")
```

For n=200 the two are almost identical (the t-distribution converges to normal as n grows). For n<30 the t-distribution CI is noticeably wider — which is exactly why people invented it.
</details>

### Exercise 3 — ⭐⭐ Sample size planning

You want to detect a **0.3 effect size** with 90% power and significance 0.05. How many users per arm do you need? What if you can tolerate 80% power instead?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
print(f"d = 0.3, power = 0.80 → n = {sample_size_for_d(0.3, power=0.80):,} per arm")
print(f"d = 0.3, power = 0.90 → n = {sample_size_for_d(0.3, power=0.90):,} per arm")
print(f"d = 0.5, power = 0.80 → n = {sample_size_for_d(0.5, power=0.80):,} per arm")
```

Doubling the power from 80% to 90% requires ~30% more data — a real cost. The
practical takeaway: 80% power is the standard convention for a reason; pushing
to 90% costs a lot of users.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

The snippet below is supposed to compute the standard error of the mean, but the answer is consistently a factor of √n too small. Find the bug.

```python
def sem_buggy(data):
    return np.std(data, ddof=1) * np.sqrt(len(data))
```

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
# Your fixed version  👇


<details>
<summary>💡 <b>Solution</b></summary>

The formula for the *standard error of the mean* is `σ / √n`, not `σ × √n`.

```python
def sem(data):
    return np.std(data, ddof=1) / np.sqrt(len(data))
```

Sanity check: as `n` grows, the standard error should *shrink* (we're more sure of the mean). The buggy version grows instead — the immediate signal that the formula is upside down.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise C — ⭐⭐⭐ Permutation test for the difference of two means

Instead of assuming the data is normal (which is what a t-test does), a **permutation test** estimates the p-value purely from the data. The procedure:

1. Compute the observed difference of means `obs = mean(a) - mean(b)`.
2. Pool `a` and `b`, then repeatedly shuffle and re-split into two groups of the original sizes.
3. The p-value is the fraction of shuffled differences whose absolute value is ≥ `|obs|`.

Test on:

```python
rng = np.random.default_rng(0)
a = rng.normal(loc=10.5, scale=2.0, size=50)
b = rng.normal(loc=10.0, scale=2.0, size=50)
```

Use 2,000 permutations and report `obs`, the p-value, and how it compares to a Welch's t-test on the same data.

In [ ]:
# Your code here  👇
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
a = rng.normal(loc=10.5, scale=2.0, size=50)
b = rng.normal(loc=10.0, scale=2.0, size=50)

def permutation_p_value(a, b, n_iter=2_000, rng=None):
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
a = rng.normal(loc=10.5, scale=2.0, size=50)
b = rng.normal(loc=10.0, scale=2.0, size=50)

def permutation_p_value(a, b, n_iter=2_000, rng=None):
    rng = rng or np.random.default_rng(0)
    a, b = np.asarray(a, float), np.asarray(b, float)
    obs = a.mean() - b.mean()
    pooled = np.concatenate([a, b])
    n_a = len(a)
    count = 0
    for _ in range(n_iter):
        rng.shuffle(pooled)
        diff = pooled[:n_a].mean() - pooled[n_a:].mean()
        if abs(diff) >= abs(obs):
            count += 1
    return obs, count / n_iter

obs, p_perm = permutation_p_value(a, b)
t, p_welch = stats.ttest_ind(a, b, equal_var=False)
print(f"observed difference: {obs:.3f}")
print(f"permutation p-value: {p_perm:.4f}")
print(f"Welch t-test p-value: {p_welch:.4f}")
```

**Reasoning.** Three things worth absorbing. (1) **Permutation tests make no distributional assumptions** — they only assume the two groups are *exchangeable under the null hypothesis*. That's a much weaker assumption than 'normally distributed' and is robust to outliers and skew. (2) The p-value is *literally* the fraction of shuffles that look at least as extreme as what you observed — it has an intuitive frequency interpretation, no Student-t tables required. (3) For roughly-normal data the permutation p-value and Welch's p-value will agree closely; the divergence appears with heavy tails and skew, where the permutation test is the more honest answer.
</details>

### Stretch exercise D — ⭐⭐⭐ Two-sample t-test from summary stats

Often you have only group summaries, not raw data. Implement Welch's t-test (unequal variances) given the **means, standard deviations, and counts** of two groups, and report the p-value.

Test data:

```python
x1, s1, n1 = 100.0, 12.0, 50
x2, s2, n2 = 105.0, 14.0, 60
```

In [ ]:
# Your code here  👇
import math
from scipy import stats

x1, s1, n1 = 100.0, 12.0, 50
x2, s2, n2 = 105.0, 14.0, 60

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import math
from scipy import stats

x1, s1, n1 = 100.0, 12.0, 50
x2, s2, n2 = 105.0, 14.0, 60

se = math.sqrt(s1**2/n1 + s2**2/n2)
t  = (x1 - x2) / se

# Welch–Satterthwaite degrees of freedom
num  = (s1**2/n1 + s2**2/n2) ** 2
den  = (s1**2/n1)**2 / (n1 - 1) + (s2**2/n2)**2 / (n2 - 1)
df   = num / den

p_two_sided = 2 * stats.t.sf(abs(t), df)
print(f"t = {t:.3f}  df = {df:.2f}  p = {p_two_sided:.4f}")
```

**Reasoning.** This is the calculation `scipy.stats.ttest_ind_from_stats` does for you — but writing it once builds the right intuition. Three points. (1) Welch's formula doesn't pool the variances, which is what makes it robust to unequal variances and unequal `n`. (2) The **Welch–Satterthwaite** degrees-of-freedom approximation is the messy fraction in the middle; you almost never compute it by hand outside of textbook exercises. (3) `stats.t.sf(...)` is the *survival function* (1 − CDF), which is more numerically stable than `1 - stats.t.cdf(...)` in the tails — useful when p is tiny.
</details>

## 🧠 Key takeaways

1. **Plot before you summarise.** Histograms and box plots tell you the shape; means and standard deviations don't.
2. For **skewed or heavy-tailed** data, report median and percentiles (p50, p95, p99) *alongside* the mean.
3. A **confidence interval** turns a point estimate into honest uncertainty. The half-width shrinks with √n, so doubling precision needs 4× the data.
4. A **t-test** answers "is the difference real?". An **effect size** (Cohen's *d*) answers "is it big?". You need both.
5. **Sample-size planning** prevents the silent failure of an underpowered test.
6. The three classical mistakes are **peeking, multiple comparisons, and reporting only p**. The fixes are pre-registration, correction, and effect size.
7. Every A/B report you ship should have: sample sizes, means, difference + CI, Cohen's *d*, p-value, *and* a verdict.

## ✅ Self-assessment

- [ ] Plot a histogram + box plot and explain skewness, tails, outliers
- [ ] Compute mean, median, p95, p99 for a sample
- [ ] Compute a 95% CI for a mean
- [ ] Run a Welch's t-test and correctly interpret the p-value
- [ ] Compute and interpret Cohen's d
- [ ] Plan a sample size for a target effect size and power
- [ ] Spot the three classical A/B mistakes

## 🚀 Next step

Continue with **Notebook 8 (fast track) — Scikit-Learn Basics**, where everything so far converges: the tables, charts, and statistics become features for training and evaluating your first machine-learning models.